# 02 — Plant Entity Reconciliation

Show the cross-source matching process, OSUKED coverage, and fuzzy matching for unmatched plants.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from pathlib import Path
from uk_energy.config import PLANTS_UNIFIED, PROCESSED_DIR

## 1. Load Unified Plants

In [ ]:
if PLANTS_UNIFIED.exists():
    df = pd.read_parquet(PLANTS_UNIFIED)
    print(f'Unified plants: {len(df):,}')
    print(f'Columns: {list(df.columns)}')
    display(df.head(10))
else:
    print('plants_unified.parquet not found — run: python -m uk_energy reconcile')

## 2. Source Coverage Analysis

In [ ]:
if 'df' in dir():
    source_cols = [c for c in df.columns if c.startswith('source_')]
    coverage = {}
    for col in source_cols:
        coverage[col.replace('source_', '')] = df[col].sum()
    
    import plotly.express as px
    cov_df = pd.DataFrame(list(coverage.items()), columns=['source', 'plants_covered'])
    fig = px.bar(cov_df, x='source', y='plants_covered', title='Plants Covered by Each Source',
                 color='plants_covered', color_continuous_scale='Viridis')
    fig.show()
    display(cov_df)

## 3. Coordinate Coverage

In [ ]:
if 'df' in dir():
    has_coords = df['lat'].notna() & df['lon'].notna()
    print(f'Plants with coordinates: {has_coords.sum():,} / {len(df):,} ({has_coords.mean()*100:.1f}%)')
    print(f'Missing coordinates: {(~has_coords).sum():,}')
    
    # Missing by source
    print('\nMissing coordinates by primary source:')
    for col in [c for c in df.columns if c.startswith('source_')]:
        subset = df[df[col] == True]
        n_missing = subset['lat'].isna().sum()
        src = col.replace('source_', '')
        print(f'  {src}: {n_missing}/{len(subset)} missing ({n_missing/max(1,len(subset))*100:.0f}%)')

## 4. Capacity Distribution

In [ ]:
if 'df' in dir():
    import plotly.express as px
    
    df_with_cap = df[df['capacity_mw'].notna() & (df['capacity_mw'] > 0)]
    print(f'Plants with capacity data: {len(df_with_cap):,}')
    
    fig = px.histogram(
        df_with_cap,
        x='capacity_mw',
        color='fuel_type',
        nbins=100,
        log_x=True,
        title='Capacity Distribution by Fuel Type',
        labels={'capacity_mw': 'Capacity (MW, log scale)'}
    )
    fig.show()

## 5. Fuel Type Summary

In [ ]:
if 'df' in dir():
    fuel_summary = (
        df.groupby('fuel_type')
        .agg(
            plant_count=('plant_id', 'count'),
            total_capacity_mw=('capacity_mw', 'sum'),
            mean_capacity_mw=('capacity_mw', 'mean'),
        )
        .round(1)
        .sort_values('total_capacity_mw', ascending=False)
    )
    display(fuel_summary)

## 6. Status Breakdown

In [ ]:
if 'df' in dir() and 'status' in df.columns:
    status_summary = df.groupby('status').agg(
        count=('plant_id', 'count'),
        total_capacity_mw=('capacity_mw', 'sum')
    ).round(1)
    display(status_summary)